<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/prepare/prepare_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prepare a model and its engine on a cheap CPU runtime

Runtime → Change runtime type → **CPU** (High-RAM is not needed). This notebook does the slow, GPU-free work so the expensive GPU runtime only loads and serves:

- **Model:** downloaded from Hugging Face straight to `MyDrive/llm_models/models/<id>/`, one file at a time (local disk only ever holds two files). Re-running resumes; files already on Drive are skipped. Google Drive accepts about 750 GB of uploads per day, so the largest models may need two days.
- **Engine:** built or cached on Drive for the GPU you pick: wheels (vLLM, SGLang, Diffusers), HyperQwen's patched venv, llama.cpp / NInfer compiled for that GPU's architecture (nvcc does not need the GPU), release downloads (Strata, Maya, TensorFold, ComfyUI, Sparsa).
- Both run at the same time: the download uses the network, the build uses the CPU.

Then open `notebooks/<engine>/<engine>_gpu.ipynb` on the GPU runtime and pick the same model: it restores both from Drive.

Can also be run without a browser: `python -m colab_llm_hub prepare --target g4 --engine ninfer --model recommended`.


## 1. Mount Drive, create the workspace, fetch (or restore) the hub code

Each run fetches the latest hub code from GitHub and refreshes its copy on Drive; only if GitHub cannot be reached does it fall back to that Drive copy. Settings and tool binaries are restored from Drive.


In [ ]:
import os, re, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

ENGINE   = "prepare"  # settings go to config/settings_prepare.json
HUB_REPO  = "https://github.com/Biegle-W/google_colab_llm"
HUB_REF   = "main"  # always clone hub package from main
DRIVE_DIR = "/content/drive/MyDrive/llm_models"
HUB_DIR   = "/content/hub_repo"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

OFFLINE_HUB = False   # True = skip GitHub and use the hub code saved on Drive

os.makedirs(DRIVE_DIR, exist_ok=True)
sys.path.insert(0, HUB_DIR)
ref_key = (HUB_REF or "default").replace("/", "_")
cached = f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/colab_llm_hub"
have_cache = os.path.isdir(cached) and os.path.isfile(f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/COMPLETE")
fresh = False
if not OFFLINE_HUB:
    try:                          # always prefer the latest code: fixes land without reopening
        sh(f"rm -rf {HUB_DIR}")
        if HUB_REF and re.fullmatch(r"[0-9a-f]{40}", HUB_REF):
            sh(f"mkdir -p {HUB_DIR} && cd {HUB_DIR} && git init -q && git fetch -q --depth 1 {HUB_REPO} {HUB_REF} && git checkout -q FETCH_HEAD")
        else:
            sh(f"git clone -q --depth 1 {'--branch ' + HUB_REF if HUB_REF else ''} {HUB_REPO} {HUB_DIR}")
        fresh = True
        print("Fetched hub code from GitHub:", sh(f"cd {HUB_DIR} && git rev-parse --short HEAD"))
    except RuntimeError as e:
        print("Could not fetch from GitHub; using the copy on Drive.", str(e)[-300:])
if not fresh:
    if not have_cache:
        raise RuntimeError("No hub code: GitHub was unreachable and there is no copy on Drive yet.")
    shutil.rmtree(HUB_DIR, ignore_errors=True)
    os.makedirs(HUB_DIR, exist_ok=True)
    shutil.copytree(cached, f"{HUB_DIR}/colab_llm_hub")
    print("Restored hub package from Drive (it may be older than GitHub)")

from colab_llm_hub.workspace import open_workspace
from colab_llm_hub import install
WS = open_workspace(DRIVE_DIR, engine=ENGINE)
print("Drive workspace:", WS.describe())
# Release this runtime after this many minutes with no cell running, no API request, and no
# download / build / GPU work, so an idle VM stops using compute units (0 = never).
IDLE_TIMEOUT_MIN = 10  # @param {type:"integer"}
from colab_llm_hub import idle_guard
idle_guard.start(IDLE_TIMEOUT_MIN)
print(f"Idle timeout: {IDLE_TIMEOUT_MIN} min" if IDLE_TIMEOUT_MIN > 0 else "Idle timeout: off")
if fresh:                         # keep the Drive copy current for offline fallback
    install.cache_hub_package(HUB_DIR, str(WS.builds), ref=HUB_REF or "default")
import colab_llm_hub
print("hub", colab_llm_hub.__version__, "engine", ENGINE)


## 2. Check this runtime

Compiling llama.cpp or NInfer is limited by CPU cores, and the GPU runtime restores Python wheels and HyperQwen's venv only if its Python version matches this one.


In [ ]:
import platform, shutil
print("CPU threads :", os.cpu_count())
print("Python      :", platform.python_version(), "(the GPU runtime must match for wheels / venvs)")
print("Free disk   :", round(shutil.disk_usage("/content").free / 1e9), "GB in /content")
nvcc = subprocess.run("PATH=$PATH:/usr/local/cuda/bin nvcc --version | tail -2", shell=True,
                      capture_output=True, text=True).stdout.strip()
print("nvcc        :", nvcc or "not found (fine: llama.cpp is then built by the GPU notebook)")
if shutil.which("nvidia-smi"):
    print("Note: this runtime has a GPU. Preparing works here too, but a CPU runtime costs far less.")


## 3. Pick the GPU runtime, engine and model


In [ ]:
from colab_llm_hub import catalog, prepare, runtimes

TARGET = "g4"  # @param ["l4", "a100", "g4"]
ENGINE_TO_PREP = "vllm"  # @param ["llama", "vllm", "sglang", "diffusers", "strata", "maya", "tensorfold", "comfyui", "sparsa", "hyperqwen", "ninfer"]
MODEL = "recommended"  # @param ["recommended", "qwen3.8-27b-q4", "qwen3.8-27b-q8", "qwen3.8-27b-bf16", "qwen3.6-35b-a3b-iq4", "qwen3.6-35b-a3b-q8", "sage-a1.5-q4", "nemotron-3.5-super-120b-q3", "nemotron-3.5-super-120b-q4", "ornith-1.5-122b-q4", "ornith-1.5-122b-q5", "qwen3-14b-fp8", "qwen3.8-27b-fp8", "qwen3.8-27b-bf16-vllm", "qwen3.6-35b-a3b-fp8", "sage-a1.5-fp8", "nemotron-3.5-bolt-74b-w4a16", "sd15", "sdxl-base", "sdxl-turbo", "flux-schnell", "qwen3.8-27b-w4a16", "qwen3.8-27b-w4a16-sglang", "flash-next-iq1s", "flash-next-q3", "qwen-image-2.1", "qwen-image-2.1-nf4", "minimax-h3", "glm-5.3-flash-iq1s", "glm-5.3-flash-iq3", "deepseek-v4.1-flash-1017-iq1s", "flash-next-strata-q2", "flash-next-strata-iq3-xxs", "flash-next-strata-iq3-s", "deepseek-strata-ist-2.00", "glm-maya-s24", "glm-maya-s", "glm-maya-xs", "deepseek-maya-xs", "deepseek-maya-s", "qwen3.8-27b-exl3-2bpw", "qwen-image-2.1-turbo-comfy", "flux3-klein-comfy", "wan22-5b-comfy", "minimax-h3-comfy", "flash-next-gyro-s", "flash-next-gyro-m", "flash-next-swift-iq2-xs", "flash-next-swift-iq3-xxs", "qwen36-strata-q4", "qwen36-strata-q6", "minimax-m3.1-flash-iq2-xs-d3", "minimax-m3.1-flash-iq2-kt", "nemotron-3.5-bolt-74b-iq2", "hyperqwen-qwen3.8-27b-mtp", "hyperqwen-qwen3.8-27b-dflash2", "hyperqwen-qwen3.8-27b-batch", "ninfer-qwen3.8-27b-nvfp4", "ninfer-qwen3.8-27b", "ninfer-qwen3.6-35b-a3b", "ninfer-qwen3.8-flash-next"]
BUILD_ENGINE = True  # @param {type:"boolean"}
DOWNLOAD_MODEL = True  # @param {type:"boolean"}
REFRESH_ENGINE = False  # @param {type:"boolean"}
# llama.cpp / NInfer compile for 30+ minutes on a 2-thread CPU runtime and a few on the GPU runtime,
# so they are left to the GPU notebook unless this is ticked.
FORCE_COMPILE = False  # @param {type:"boolean"}

VRAM, RAM_GB, DISK_GB, ARCH = runtimes.specs(TARGET)
print(f"Target: {runtimes.target(TARGET)['name']} ({VRAM} GB VRAM, {RAM_GB} GB RAM, sm_{ARCH})\n")
for e in catalog.for_engines(catalog.load(), [ENGINE_TO_PREP]):
    reason = catalog.why_not(e, VRAM, RAM_GB, DISK_GB, ARCH)
    print(f"  {e['id']:40s} {e['size_gb']:7.1f} GB  " + ("ok" if reason is None else reason))
ENTRY = prepare.pick(ENGINE_TO_PREP, TARGET, MODEL)
print(f"\nWill prepare {ENTRY['id']} ({ENTRY['size_gb']:.1f} GB) for {ENGINE_TO_PREP} on {TARGET}")


## 4. Download the model and build the engine

Safe to stop and re-run: finished files and builds are kept on Drive. Progress prints every 30 s.


In [ ]:
import threading
from colab_llm_hub.store import ModelStore

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

store = ModelStore(DRIVE_DIR, prepare.DEFAULT_STAGING, token=os.environ.get("HF_TOKEN"))
done = threading.Event()
def report():
    while not done.wait(30):
        p = store.progress.get(ENTRY["id"])
        if p:
            print(f"  {ENTRY['id']}: {p['done_bytes'] / 1e9:.1f} / {p['total_bytes'] / 1e9:.1f} GB on Drive")
threading.Thread(target=report, daemon=True).start()
try:
    RESULT = prepare.prepare(TARGET, ENGINE_TO_PREP, ENTRY["id"], drive_dir=DRIVE_DIR, store=store,
                             do_engine=BUILD_ENGINE, do_download=DOWNLOAD_MODEL, refresh=REFRESH_ENGINE,
                             force_compile=FORCE_COMPILE)
finally:
    done.set()
print(RESULT)
print("Install log →", WS.logs / "install.log")


## 5. Next

Disconnect this runtime (Runtime → Disconnect and delete runtime) so it stops using compute units. Then switch to the GPU runtime, open `notebooks/<engine>/<engine>_gpu.ipynb` and pick the same model.
